In [962]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation


class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_ext_id,HC_inj_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_ext_id=HC_ext_id
        self.HC_inj_id=HC_inj_id
        self.net=net
        self._build_tespy_Cycles()

    def _build_tespy_Cycles(self):
        def my_ude(ude):
            dt_DHN=ude.params['dt']
            return ude.conns[0].calc_T()+dt_DHN-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        self.nw_Cooling_net = Network()
        self.nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Cooling_net_compressor = Compressor("compresor")
        self.Cooling_net_condenser = Condenser("condensador")
        self.Cooling_net_valve = Valve("valvula_expansion")
        self.Cooling_net_evaporator = HeatExchanger("evaporador")
        self.Cooling_net_cc=CycleCloser('CycleCloser')
        self.Cooling_net_source_consumer=Source("Source_Consumer ")
        self.Cooling_net_sink_consumer=Sink("Sink_consumer")
        self.Cooling_net_source_reseau=Source("Source_Reseau")
        self.Cooling_net_sink_reseau=Sink("Sink_Reseau")
        self.Cooling_net_c0=Connection(self.Cooling_net_valve, 'out1', self.Cooling_net_cc, 'in1', label='0')
        self.Cooling_net_c1 = Connection(self.Cooling_net_cc, 'out1', self.Cooling_net_evaporator, 'in2', label='1')
        self.Cooling_net_c2 = Connection(self.Cooling_net_evaporator, 'out2', self.Cooling_net_compressor, 'in1', label='2')
        self.Cooling_net_c3 = Connection(self.Cooling_net_compressor, 'out1', self.Cooling_net_condenser, 'in1', label='3')
        self.Cooling_net_c4 = Connection(self.Cooling_net_condenser, 'out1', self.Cooling_net_valve, 'in1', label='4')
        self.Cooling_net_c5=Connection(self.Cooling_net_condenser, 'out2',self.Cooling_net_sink_consumer, 'in1', label='5')
        self.Cooling_net_c6=Connection(self.Cooling_net_source_consumer, 'out1',self.Cooling_net_condenser , 'in2', label='6')
        self.Cooling_net_c7=Connection(self.Cooling_net_source_reseau, 'out1',self.Cooling_net_evaporator , 'in1', label='7')
        self.Cooling_net_c8=Connection(self.Cooling_net_evaporator, 'out1',self.Cooling_net_sink_reseau , 'in1', label='8')
        self.nw_Cooling_net.add_conns(self.Cooling_net_c0,  self.Cooling_net_c1,  self.Cooling_net_c2,  self.Cooling_net_c3,  self.Cooling_net_c4 , self.Cooling_net_c5,  self.Cooling_net_c6, self.Cooling_net_c7, self.Cooling_net_c8)
        self.Cooling_ude = UserDefinedEquation(
                    'my_ude', my_ude, my_ude_dependents, conns=[self.Cooling_net_c8, self.Cooling_net_c7],params={'dt': 5})
        self.nw_Cooling_net.add_ude(self.Cooling_ude)
        
        #Heating net 
        self.nw_Heating_net = Network()
        self.nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Heating_net_compressor = Compressor("compresor")
        self.Heating_net_condenser = Condenser("condensador")
        self.Heating_net_valve = Valve("valvula_expansion")
        self.Heating_net_evaporator = HeatExchanger("evaporador")
        self.Heating_net_cc=CycleCloser('CycleCloser')
        self.Heating_net_source_consumer=Source("Source_Consumer ")
        self.Heating_net_sink_consumer=Sink("Sink_consumer")
        self.Heating_net_source_reseau=Source("Source_Reseau")
        self.Heating_net_sink_reseau=Sink("Sink_Reseau")
        self.Heating_net_c0=Connection(self.Heating_net_valve, 'out1', self.Heating_net_cc, 'in1', label='0')
        self.Heating_net_c1 = Connection(self.Heating_net_cc, 'out1', self.Heating_net_evaporator, 'in2', label='1')
        self.Heating_net_c2 = Connection(self.Heating_net_evaporator, 'out2', self.Heating_net_compressor, 'in1', label='2')
        self.Heating_net_c3 = Connection(self.Heating_net_compressor, 'out1', self.Heating_net_condenser, 'in1', label='3')
        self.Heating_net_c4 = Connection(self.Heating_net_condenser, 'out1', self.Heating_net_valve, 'in1', label='4')
        self.Heating_net_c5 = Connection(self.Heating_net_evaporator, 'out1', self.Heating_net_sink_consumer, 'in1', label='5')
        self.Heating_net_c6 = Connection(self.Heating_net_source_consumer, 'out1', self.Heating_net_evaporator, 'in1', label='6')  
        self.Heating_net_c7 = Connection(self.Heating_net_source_reseau, 'out1', self.Heating_net_condenser, 'in2', label='7')
        self.Heating_net_c8 = Connection(self.Heating_net_condenser, 'out2', self.Heating_net_sink_reseau, 'in1', label='8')
        self.nw_Heating_net.add_conns(self.Heating_net_c0,  self.Heating_net_c1,  self.Heating_net_c2,  self.Heating_net_c3,  self.Heating_net_c4 , self.Heating_net_c5,  self.Heating_net_c6, self.Heating_net_c7, self.Heating_net_c8)
        self.Heating_ude = UserDefinedEquation(
                        'my_ude_4', my_ude, my_ude_dependents, conns=[self.Heating_net_c7, self.Heating_net_c8],params={'dt': 5})
        self.nw_Heating_net.add_ude(self.Heating_ude)
        
    def solve_cycle(self,mode,Q_consumer,eta_s,T_nework_in,T_cons,dt_DHN):
        #The pressure values for the consumer and district heating side in the heating pump are arbitrary values
        self.mode=mode
        T_cons_in = T_cons[0]
        T_cons_out = T_cons[1]
        T_DH_in = T_nework_in
        if self.mode=="COOLING_NET":
            self.Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5,Q=Q_consumer)
            self.Cooling_net_compressor.set_attr(eta_s=eta_s)
            self.Cooling_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Cooling_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Cooling_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Cooling_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Cooling_net._set_p_range([p_triple, p_high])
            self.nw_Cooling_net._set_h_range([h_min,h_max])
            self.Cooling_ude.params['dt'] = dt_DHN
            self.nw_Cooling_net.solve('design')        
        elif self.mode=="HEATING_NET":
            self.Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5,Q=Q_consumer)
            self.Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Heating_net_compressor.set_attr(eta_s=eta_s)
            self.Heating_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Heating_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Heating_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Heating_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Heating_net._set_p_range([p_triple, p_high])
            self.nw_Heating_net._set_h_range([h_min,h_max])
            self.Heating_ude.params['dt'] = dt_DHN
            self.nw_Heating_net.solve('design')
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_ext_id, "qext_w"] =abs( self.Cooling_net_evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_ext_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_inj_id, "qext_w"] = self.Heating_net_condenser.Q.val
            self.net.heat_consumer.at[self.HC_inj_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False

        else: 
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False

    def get_main_results(self):
        if self.mode=="COOLING_NET":
            results = {
                "Q_consumer": abs(self.Cooling_net_condenser.Q.val),
                "Q_network": abs(self.Cooling_net_evaporator.Q.val),
                "W_compressor":self.Cooling_net_compressor.P.val ,
                "COP": abs(self.Cooling_net_condenser.Q.val)/self.Cooling_net_compressor.P.val 
            }
        elif self.mode=="HEATING_NET":
            results = {
                "Q_consumer": abs(self.Heating_net_evaporator.Q.val),
                "Q_network": abs(self.Heating_net_condenser.Q.val),
                "COP": abs(self.Heating_net_evaporator.Q.val)/self.Heating_net_compressor.P.val 
            }
        else:
            results = {}
        return results

    def get_results_solver(self):
        if self.mode=="COOLING_NET":
            self.nw_Cooling_net.print_results()
        elif self.mode=="HEATING_NET":
            self.nw_Heating_net.print_results()
        else:
            return None


In [963]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_Ida")
j_HP_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_Ida")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_ret")

#Plant-Nodo
u=0.35 / (np.pi * 0.15)
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u, text_k=273.2, name="Tubo_Ida_Plant_Nodo_1",k_mm=0.1*1000
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_RetornoPlant_Nodo_1",k_mm=0.1*1000)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="nodo_1_nodo_2",k_mm=0.1*1000
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="nodo_2_ret_nodo_1_ret",k_mm=0.1*1000
)

#Node HP
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_1_ida",k_mm=0.1*1000
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_1_ret",k_mm=0.1*1000
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_2_ret",k_mm=0.1*1000
)

#Heat Pumps
#Heat pump: extraction
HP_1_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_1 ,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat mump 1: Injection
HP_1_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_1,
    to_junction=j_HP_1,
    qext_w=-150000,
    deltat_k=50,
    name="HP_1_INJ"
)
#Heat pump: extraction
HP_2_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_2 ,
    to_junction=j_HP_ret_2,
    qext_w=150000,
    deltat_k=50,
    name="HP_2_EXT"
)
#Heat mump 1: Injection
HP_2_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_2,
    to_junction=j_HP_2,
    qext_w=150000,
    deltat_k=-50,
    name="HP_2_INJ"
)
#Plant: 

Plant_1_ida = pp.create_ext_grid(
    net, junction=j_fuente_ida, p_bar=3.0, t_k=340,
    type="pt", name="Grid_ida",in_service=False
)
Plant_1_ret = pp.create_ext_grid(
    net, junction=j_fuente_ret, p_bar=2.5,   # = p_flow_bar - plift_bar, para conservar el mismo salto de presión
    type="p", name="Grid_ret",in_service=False
)
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=340 ,name='Grid')

In [964]:
T_network_inital_guess_1=35
T_network_inital_guess_2=50


In [965]:
HP_1=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_ext_id=HP_1_Ext,HC_inj_id=HP_1_inj)
HP_2=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_ext_id=HP_2_Ext,HC_inj_id=HP_2_inj)

In [966]:
T_network_HP_1=T_network_inital_guess_1
T_network_HP_2=T_network_inital_guess_2
T_network_HP_1_loop=T_network_inital_guess_1
T_network_HP_2_loop=T_network_inital_guess_2
tolerance=1E-6
error_1=10
error_2=10
iterations=0
while error_1>tolerance or error_2>tolerance:
    HP_1.solve_cycle(mode="COOLING_NET",Q_consumer=-15000,eta_s=0.95,T_nework_in=T_network_HP_1,T_cons=[50,60],dt_DHN=5)
    HP_2.solve_cycle(mode="HEATING_NET",Q_consumer=-150000,eta_s=0.95,T_nework_in=T_network_HP_2,T_cons=[35,25],dt_DHN=10)
    print("iteration")
    HP_1.interaction_simulation(dT_water=5)
    HP_2.interaction_simulation(dT_water=-10)
    try:
        pp.pipeflow(net,mode="bidirectional") 
    except UserWarning: 
        net.circ_pump_pressure.at[Plant_1, "in_service"] =False
        net.ext_grid.at[Plant_1_ida, "in_service"] =True
        net.ext_grid.at[Plant_1_ret, "in_service"] =True 
        pp.pipeflow(net,mode="bidirectional")
        net.circ_pump_pressure.at[Plant_1, "in_service"] =True
        net.ext_grid.at[Plant_1_ida, "in_service"] =False
        net.ext_grid.at[Plant_1_ret, "in_service"] =False
    T_network_HP_1=float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_from_k"]-273.15)
    T_network_HP_2=float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_from_k"]-273.15)
    error_1=abs(T_network_HP_1_loop-T_network_HP_1)
    error_2=abs(T_network_HP_2_loop-T_network_HP_2)
    print(error_1)
    print(error_2)
    T_network_HP_1_loop=T_network_HP_1
    T_network_HP_2_loop=T_network_HP_2
    iterations+=1


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 9.04e+05   | 0 %        | 1.74e+00   | 1.42e+06   | 4.03e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 3.23e+04   | 16 %       | 9.74e-01   | 1.97e+05   | 6.96e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 4.66e+03   | 25 %       | 2.22e-01   | 1.64e+04   | 4.58e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 1.85e+00   | 63 %       | 1.18e-05   | 1.58e+02   | 4.33e+00   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 1.77e-04   | 100 %      | 8.99e-10   | 1.41e-02   | 3.71e-04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 3.47e-10   | 100 %      | 5.48e-13   | 4.94e-09   | 1.90e-08   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 6, Calculation time: 0.02 s, Iterations per second: 241.20

 iter  | residual   | pro

In [967]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,14557.13447,NaN,5.0,NaN,True,heat_consumer
1,HP_1_INJ,8,3,-150000.00000,NaN,50.0,NaN,False,heat_consumer
2,HP_2_EXT,4,9,150000.00000,NaN,50.0,NaN,False,heat_consumer
3,HP_2_INJ,9,4,-192123.90204,NaN,-10.0,NaN,True,heat_consumer


In [968]:
net.junction

,name,pn_bar,tfluid_k,height_m,in_service,type
0,Fuente_Ida,5.0,353.15,0.0,True,junction
1,nodo_1_ida,5.0,353.15,0.0,True,junction
2,nodo_2_ida,5.0,353.15,0.0,True,junction
3,HP_1_Ida,5.0,353.15,0.0,True,junction
4,HP_2_Ida,5.0,353.15,0.0,True,junction
5,Fuente_Retorno,1.5,333.15,0.0,True,junction
6,Node_1_ret,5.0,353.15,0.0,True,junction
7,Nodo_2_ret,5.0,353.15,0.0,True,junction
8,HP_1_ret,5.0,353.15,0.0,True,junction
9,HP_2_ret,5.0,353.15,0.0,True,junction


In [969]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,14557.13447,NaN,5.0,NaN,True,heat_consumer
1,HP_1_INJ,8,3,-150000.00000,NaN,50.0,NaN,False,heat_consumer
2,HP_2_EXT,4,9,150000.00000,NaN,50.0,NaN,False,heat_consumer
3,HP_2_INJ,9,4,-192123.90204,NaN,-10.0,NaN,True,heat_consumer


In [970]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,3.086833,2.413631,339.867145,334.867145,334.867145,0.695295,-0.695295,0.000709,5.000000,14557.13447
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.717145,-150000.00000
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.055906,150000.00000
3,2.381657,3.118968,330.205906,340.205906,340.205906,4.589201,-4.589201,0.004674,-10.000000,-192123.90204


In [971]:
net.circ_pump_pressure

,name,return_junction,flow_junction,p_flow_bar,t_flow_k,plift_bar,in_service,type
0,Grid,5,0,3.0,340.0,0.5,True,pt


In [972]:
net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.452736,NaN


In [973]:
net.ext_grid

,name,junction,p_bar,t_k,in_service,type
0,Grid_ida,0,3.0,340.0,False,pt
1,Grid_ret,5,2.5,NaN,False,p


In [974]:
net.res_ext_grid

,mdot_kg_per_s
0,3.893906
1,-3.893906


In [975]:
net.res_junction

,p_bar,t_k
0,3.000000,340.000000
1,3.043866,340.084483
2,3.087734,340.169072
3,3.086833,339.867145
4,3.118968,340.205906
5,2.500000,329.602736
6,2.456366,329.531410
7,2.412733,330.237295
8,2.413631,334.867145
9,2.381657,330.205906


In [976]:
net.pipe

,name,from_junction,to_junction,std_type,length_km,inner_diameter_mm,outer_diameter_mm,k_mm,loss_coefficient,u_w_per_m2k,text_k,sections,in_service,type
0,Tubo_Ida_Plant_Nodo_1,0,1,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
1,Tubo_RetornoPlant_Nodo_1,6,5,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
2,nodo_1_nodo_2,1,2,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
3,nodo_2_ret_nodo_1_ret,7,6,None,0.0589,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
4,Tubo_Ida_nodo_2_cons_1_ida,2,3,None,0.0376,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
5,Tubo_Retorno_nodo_2_cons_1_ret,8,7,None,0.0376,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
6,Tubo_Ida_nodo_2_cons_2_ida,2,4,None,0.0302,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe
7,Tubo_Retorno_nodo_2_cons_2_ret,9,7,None,0.0302,150.0,150.0,100.0,0.0,0.742723,273.2,1,True,pipe


In [977]:
net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,-0.224981,3.000000,3.043866,340.000000,340.084483,340.000000,-3.893906,3.893906,-0.003976,78480.613861,0.450684,-0.043866
1,-0.223726,2.456366,2.500000,329.531410,329.602736,329.531410,-3.893906,3.893906,-0.003954,67124.811598,0.450822,-0.043634
2,-0.224992,3.043866,3.087734,340.084483,340.169072,340.084483,-3.893906,3.893906,-0.003976,78572.087730,0.450683,-0.043868
3,-0.223718,2.412733,2.456366,330.237295,329.531410,329.460174,-3.893906,3.893906,-0.003953,67052.982421,0.450823,-0.043633
4,0.040172,3.087734,3.086833,340.169072,339.867145,339.867145,0.695295,-0.695295,0.000710,14008.820523,0.454437,0.000900
5,0.040055,2.413631,2.412733,334.867145,330.237295,334.588941,0.695295,-0.695295,0.000708,12963.985959,0.454805,0.000898
6,-0.265176,3.087734,3.118968,340.169072,340.205906,340.169072,-4.589201,4.589201,-0.004686,92679.477027,0.450559,-0.031235
7,-0.263762,2.381657,2.412733,330.205906,330.237295,330.205906,-4.589201,4.589201,-0.004661,79896.496832,0.450669,-0.031076


In [978]:
net.junction

,name,pn_bar,tfluid_k,height_m,in_service,type
0,Fuente_Ida,5.0,353.15,0.0,True,junction
1,nodo_1_ida,5.0,353.15,0.0,True,junction
2,nodo_2_ida,5.0,353.15,0.0,True,junction
3,HP_1_Ida,5.0,353.15,0.0,True,junction
4,HP_2_Ida,5.0,353.15,0.0,True,junction
5,Fuente_Retorno,1.5,333.15,0.0,True,junction
6,Node_1_ret,5.0,353.15,0.0,True,junction
7,Nodo_2_ret,5.0,353.15,0.0,True,junction
8,HP_1_ret,5.0,353.15,0.0,True,junction
9,HP_2_ret,5.0,353.15,0.0,True,junction


In [979]:
results_1=HP_1.get_main_results()
results_2=HP_2.get_main_results()

In [980]:
print(results_1)

{'Q_consumer': 15000.0, 'Q_network': 14557.134469994498, 'W_compressor': 442.8655300055088, 'COP': 33.87032628123804}


In [981]:
15000.0/1685.6799401256844

8.898486386971895

In [982]:
print(results_2)

{'Q_consumer': 150000.0, 'Q_network': 192123.90204016608, 'COP': 3.560923673618165}


In [983]:
HP_1.get_results_solver()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 4.43e+02 | 1.22e+00 | -3.35e+00 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  

In [984]:
HP_2.get_results_solver()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 4.21e+04 | 3.88e+00 | -1.64e+01 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  